# 13 · Orchestration & Operations

A pipeline isn't done until it runs **automatically** — on schedule, in the right
order, with retries and alerts, without anyone babysitting it. That's
**orchestration**. On Databricks the native tool is **Workflows / Lakeflow Jobs**.

You already built the pipeline steps (ingest → Silver → Gold, notebooks 7/9/10).
This notebook shows how to wire them into a scheduled, multi-task **job**.

## 1 · Jobs & tasks

A **Job** is a collection of **tasks** arranged as a DAG:

- Each **task** runs a notebook, Python script, SQL, a DLT pipeline, or a JAR.
- **Dependencies** define order (`transform` runs after `ingest` succeeds).
- **Schedule** — cron/interval, or trigger on file arrival.
- **Compute** — a **job cluster** spun up per run then torn down (cheaper than
  keeping a cluster on), or serverless.
- **Retries**, **timeouts**, **email/Slack alerts**, and **parameters** per task.

Our BrewBox daily job would look like:

```
ingest (nb 7)  →  transform_silver (nb 9)  →  model_gold (nb 10)
                                            ↘  (on failure) → alert
```

## 2 · Defining a job (JSON/UI)

You create jobs in the **Workflows** UI, or as JSON/YAML for version control. The
shape of a multi-task job:

```json
{
  "name": "brewbox_daily",
  "schedule": {"quartz_cron_expression": "0 0 2 * * ?", "timezone_id": "UTC"},
  "tasks": [
    {"task_key": "ingest",
     "notebook_task": {"notebook_path": "/Repos/.../7_Databricks_Data_Ingestion"}},
    {"task_key": "silver",
     "depends_on": [{"task_key": "ingest"}],
     "notebook_task": {"notebook_path": "/Repos/.../9_Databricks_Transformations_Silver"}},
    {"task_key": "gold",
     "depends_on": [{"task_key": "silver"}],
     "notebook_task": {"notebook_path": "/Repos/.../10_Databricks_Data_Modeling_Gold"}}
  ],
  "max_concurrent_runs": 1,
  "email_notifications": {"on_failure": ["data-team@example.com"]}
}
```

## 3 · Parameterize with widgets

Jobs pass **parameters** to notebooks via **widgets** — so the same notebook can
run for any date/environment. This makes pipelines reusable and backfillable.

In [ ]:
# Runs on Databricks. Creates a parameter the job (or you) can set.
try:
    dbutils.widgets.text("run_date", "2024-06-01")
    run_date = dbutils.widgets.get("run_date")
except Exception:
    run_date = "2024-06-01"   # fallback outside Databricks
print("Processing for run_date =", run_date)
# ...use run_date to filter/incrementally process, e.g. WHERE order_date = run_date

## 4 · Passing values between tasks

Tasks are isolated, but you can pass small values downstream with **task values**:

```python
# In the 'ingest' task:
dbutils.jobs.taskValues.set(key="row_count", value=1234)

# In a later task:
n = dbutils.jobs.taskValues.get(taskKey="ingest", key="row_count")
```

Great for "how many rows did we load?" gates and logging.

## 5 · CI/CD with Databricks Asset Bundles (DAB)

For production you don't click jobs together by hand — you **define them as code**
and deploy across environments (dev → prod) with **Databricks Asset Bundles**: a
`databricks.yml` describing jobs, pipelines, and clusters, deployed via the
`databricks bundle deploy` CLI (often from a Git CI pipeline).

```yaml
# databricks.yml (excerpt)
resources:
  jobs:
    brewbox_daily:
      name: brewbox_daily
      schedule: {quartz_cron_expression: "0 0 2 * * ?", timezone_id: UTC}
      tasks:
        - task_key: ingest
          notebook_task: {notebook_path: ./7_Databricks_Data_Ingestion.ipynb}
        - task_key: silver
          depends_on: [{task_key: ingest}]
          notebook_task: {notebook_path: ./9_Databricks_Transformations_Silver.ipynb}
```

This gives you version control, code review, and repeatable deploys — real
software engineering for data pipelines.

## 6 · Databricks vs external orchestrators

- **Databricks Workflows / Lakeflow Jobs** — native, no extra infra, deep
  integration (job clusters, DLT tasks, lineage). The default here.
- **Airflow / Dagster / Prefect** — use when you orchestrate *across* many systems
  (not just Databricks), or your org already standardizes on them. They trigger
  Databricks jobs via operators/the REST API.

## 7 · Exercises

**Exercise 1 —** Set a `run_date` widget and print a message using it (simulating a
job parameter).

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
try:
    dbutils.widgets.text("run_date", "2024-06-15")
    rd = dbutils.widgets.get("run_date")
except Exception:
    rd = "2024-06-15"
print(f"Pipeline would process partition for {rd}")

**Exercise 2 —** In one sentence, why is a **job cluster** usually cheaper than an
all-purpose cluster for a scheduled pipeline? (Answer in the cell.)

In [ ]:
print("A job cluster is created just for the run and torn down when it finishes, "
      "so you don't pay for idle compute between scheduled runs.")

**Exercise 3 —** Order these three tasks with dependencies for a daily job:
`model_gold`, `ingest`, `transform_silver`. (Answer in the cell.)

In [ ]:
print("ingest -> transform_silver -> model_gold  "
      "(each depends on the previous succeeding; gold runs last).")

## 8 · Recap & next

Orchestration turns your notebooks into a reliable, scheduled pipeline:
**Workflows / Lakeflow Jobs** with tasks, dependencies, schedules, retries,
parameters (**widgets**), and **task values** — deployed as code with **Asset
Bundles** for CI/CD.

**Next → `14` Performance & cost:** make the pipeline fast and cheap — Photon,
AQE, partitioning vs clustering, file sizing, and reading the Spark UI. 🚀